# density2sse — compare training runs

Requires `pandas` and `matplotlib` (e.g. `pip install -e ".[dev]"`). Load all `outputs/train/*/metrics.csv` and aggregate by `model_name`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

root = Path("../outputs/train")
runs = sorted(root.glob("*/metrics.csv"))
if not runs:
    print("No runs found under", root.resolve())
else:
    df = pd.concat([pd.read_csv(r) for r in runs], ignore_index=True)
    display(df.tail(10))

In [ ]:
if runs:
    final = df.loc[df.groupby(["run_id", "split"])["epoch"].idxmax()]
    summary = final[final["split"] == "val"].groupby("model_name")[["center_error", "angle_error", "coverage_ratio", "loss_total"]].mean()
    display(summary)

In [ ]:
if runs:
    val_df = df[df["split"] == "val"]
    for metric in ["center_error", "angle_error", "coverage_ratio"]:
        if metric in val_df.columns:
            pivot = val_df.groupby(["epoch", "model_name"])[metric].mean().unstack()
            pivot.plot(kind="line", title=metric)
            plt.xlabel("epoch")
            plt.ylabel(metric)
            plt.show()